# QUERY 3
Cuenta de origen y monto de transacciones USD en el período [2022-09-06, 2022-09-15]
con monto menor a 1 centésimo del promedio encontrado para el mismo formato de
pago en el período [2022-09-01, 2022-09-05]

In [1]:
import pandas as pd

RUTA_DE_DATASETS             = "../../datasets"
NOMBRE_ARCHIVO_COMPLETO      = "LI-Small_Trans.csv"
NOMBRE_ARCHIVO_TRANSACCIONES = "q3_sample.csv"
RUTA_RESULTADO_QUERY3        = "q3_solucion.csv"
SAMPLE_N                     = 5000  

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_COMPLETO}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)

# Filtrar USD antes de samplear
transacciones_usd_completas = transacciones_completas[
    transacciones_completas["Payment Currency"] == "US Dollar"
]

transacciones = transacciones_usd_completas.sample(n=min(SAMPLE_N, len(transacciones_usd_completas)), random_state=42)
guardar_csv(transacciones, f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}")
print(f"Sample generada: {len(transacciones)} filas USD → {NOMBRE_ARCHIVO_TRANSACCIONES}")

Sample generada: 5000 filas USD → q3_sample.csv


In [2]:
transacciones = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)
print(f"Transacciones cargadas: {len(transacciones)}")

Transacciones cargadas: 5000


In [3]:
transacciones["Timestamp"]   = pd.to_datetime(transacciones["Timestamp"])
transacciones["Amount Paid"] = pd.to_numeric(transacciones["Amount Paid"], errors="coerce")
transacciones = transacciones.dropna(subset=["Amount Paid", "Payment Format"])

periodo_temprano = transacciones[
    (transacciones["Timestamp"] >= "2022-09-01") &
    (transacciones["Timestamp"] <= "2022-09-05")
]
periodo_tardio = transacciones[
    (transacciones["Timestamp"] >= "2022-09-06") &
    (transacciones["Timestamp"] <= "2022-09-15")
]

print(f"Período temprano (1/9-5/9): {len(periodo_temprano)} transacciones")
print(f"Período tardío  (6/9-15/9): {len(periodo_tardio)} transacciones")

Período temprano (1/9-5/9): 2239 transacciones
Período tardío  (6/9-15/9): 2291 transacciones


In [4]:
stats_por_formato = (
    periodo_temprano
    .groupby("Payment Format")["Amount Paid"]
    .agg(suma="sum", count="count")
)
stats_por_formato["promedio"] = stats_por_formato["suma"] / stats_por_formato["count"]

print(stats_por_formato)

                        suma  count       promedio
Payment Format                                    
ACH             2.071944e+08    227  912750.764846
Cash            1.479459e+08    193  766559.296736
Cheque          1.653511e+08    718  230293.986365
Credit Card     1.667880e+06    521    3201.304683
Reinvestment    2.460734e+08    498  494123.249940
Wire            3.083159e+07     82  375994.970122


In [5]:
df = periodo_tardio.copy()
df["Promedio Formato"] = df["Payment Format"].map(stats_por_formato["promedio"])
df = df.dropna(subset=["Promedio Formato"])

resultado_query3 = df[
    df["Amount Paid"] < df["Promedio Formato"] * 0.01
][["Account", "Amount Paid"]].rename(columns={"Account": "From Account"}).reset_index(drop=True)

guardar_csv(resultado_query3, RUTA_RESULTADO_QUERY3)
print(f"Resultados Q3: {len(resultado_query3)} transacciones")
resultado_query3.head()

Resultados Q3: 1188 transacciones


,From Account,Amount Paid
0,8001DC570,0.82
1,80F83CD30,583.23
2,80F69ECF0,398.43
3,80087A840,2263.53
4,805E923B0,1831.17


In [6]:
# Validación del resultado
df_validacion = resultado_query3.copy()
df_validacion = df_validacion.merge(
    periodo_tardio[["Account", "Payment Format", "Timestamp", "Amount Paid"]].rename(columns={"Account": "From Account"}),
    on=["From Account", "Amount Paid"],
    how="left"
)
df_validacion["Promedio Formato"] = df_validacion["Payment Format"].map(stats_por_formato["promedio"])
df_validacion["1% Promedio"] = df_validacion["Promedio Formato"] * 0.01
df_validacion["Es valido"] = df_validacion["Amount Paid"] < df_validacion["1% Promedio"]

print(f"Filas totales: {len(df_validacion)}")
print(f"Filas válidas: {df_validacion['Es valido'].sum()}")
print(f"Filas inválidas: {(~df_validacion['Es valido']).sum()}")
df_validacion.head(10)

Filas totales: 1198
Filas válidas: 1198
Filas inválidas: 0


,From Account,Amount Paid,Payment Format,Timestamp,Promedio Formato,1% Promedio,Es valido
0,8001DC570,0.82,Wire,2022-09-10 05:27:00,375994.970122,3759.949701,True
1,80F83CD30,583.23,Cheque,2022-09-09 18:44:00,230293.986365,2302.939864,True
2,80F69ECF0,398.43,ACH,2022-09-09 15:40:00,912750.764846,9127.507648,True
3,80087A840,2263.53,Cheque,2022-09-06 06:41:00,230293.986365,2302.939864,True
4,805E923B0,1831.17,Cheque,2022-09-08 16:33:00,230293.986365,2302.939864,True
5,80D2DD1C0,1470.68,Cheque,2022-09-09 01:58:00,230293.986365,2302.939864,True
6,10042B660,507.97,Cheque,2022-09-08 22:06:00,230293.986365,2302.939864,True
7,801F84420,401.75,Cheque,2022-09-07 20:31:00,230293.986365,2302.939864,True
8,801BFC770,97.89,Cash,2022-09-09 16:58:00,766559.296736,7665.592967,True
9,805CF37C0,7931.15,ACH,2022-09-07 18:27:00,912750.764846,9127.507648,True
